[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/35.%20The%20Terminal%20Layout%20Design%20Problem/P35-Tier-3_executed.ipynb)

---


# 35. The Terminal Layout Design Problem

## Tier 3: Harmony Search

### Goal

Use Harmony Search, a population-based metaheuristic inspired by musical improvisation, to find a low-cost assignment of terminal facilities to candidate locations. The method maintains a collection of feasible assignments, creates new assignments using memory consideration, pitch adjustment, and random selection, and replaces a poor stored assignment when a better one is found.

This is a stochastic search method: it can produce useful solutions without enumerating every assignment, but it does not certify that a solution is globally optimal. Results depend on the search parameters and random seed.

### Inputs, outputs, and scope

- **Inputs:** facility area requirements, candidate-site coordinates and areas, directed facility-to-facility flows, and Harmony Search parameters.
- **Output:** the best feasible facility assignment encountered, its transportation cost, and run diagnostics such as convergence and memory diversity.
- **Scope:** sites are represented by points in a two-dimensional plane. A facility must fit the area capacity of its assigned site, and no two facilities may use the same site. The model does not include facility footprints, adjacency or safety-clearance rules, construction costs, site capacity beyond area, or uncertainty in flows and distances. Facility capacity and priority are recorded as data but are not used in this objective or constraints.

### Modeling assumptions

Coordinates are measured in meters, area requirements and site areas in square meters, flows in containers per hour, and the transportation coefficient in dollars per container-meter. The coefficient is a cost rate; multiplying it by flow and distance gives the modeled hourly transportation cost. Distances are Euclidean, and flows are directed, so the cost from facility \(i\) to facility \(j\) can differ from the cost from \(j\) to facility \(i\).

For assignment \(a=(a_1,\ldots,a_n)\), where \(a_i\) is the location index assigned to facility \(i\), the objective is

$$
C(a)=\sum_{i=1}^{n}\sum_{j=1}^{n},\;i\ne j\; f_{ij}\,d_{a_i a_j}\,r,
$$

subject to

$$
a_i\ne a_j\quad(i\ne j),\qquad A_i\le S_{a_i}\quad(i=1,\ldots,n).
$$

Here, \(f_{ij}\) is the directed flow from facility \(i\) to facility \(j\), \(d_{a_i a_j}\) is the Euclidean distance between their assigned sites, \(r\) is the transportation cost rate, \(A_i\) is facility \(i\)'s area requirement, and \(S_{a_i}\) is the area of site \(a_i\). In the code, these quantities are represented by `flow_matrix`, `calculate_distance`, `transport_cost_rate`, `Facility.area_requirement`, and `Location.area`. The objective is implemented by `calculate_total_cost`; feasibility is checked by `is_feasible_assignment`.

### Harmony Search in this notebook

A **harmony** is one complete feasible assignment. The **Harmony Memory (HM)** is the population of stored assignments. During improvisation, the algorithm considers a location used by a stored harmony with probability **HMCR** (Harmony Memory Consideration Rate); with probability **PAR** (Pitch Adjustment Rate), it instead adjusts that choice. Otherwise, it selects a feasible location at random. Locations already assigned in the new harmony are unavailable, which enforces the no-duplicate-site constraint. If a partial construction cannot be completed, a fresh feasible assignment is generated.

The memory is ranked by fitness \(F(a)=1/C(a)\), so a lower cost has a higher fitness. A new harmony replaces the worst stored harmony only when its fitness is strictly higher. The algorithm records the best cost and average pairwise Hamming distance in memory at each iteration. Hamming distance counts the facility positions at which two assignments differ; it is a measure of assignment diversity, not a transportation distance.

The search space contains at most \(m!/(m-n)!\) distinct assignments when \(n\) facilities are assigned to \(m\) distinct sites, before area-feasibility restrictions. Each iteration constructs and evaluates one candidate; this implementation also calculates memory diversity, which requires pairwise comparisons among the stored harmonies. Harmony Search has no optimality guarantee, and the stopping rule is a maximum iteration count or a fixed period without improvement.

The example uses five facilities, eight candidate sites, HMS = 20, HMCR = 0.9, PAR = 0.3, and a maximum of 1,000 iterations. These are the parameters of this notebook's example, not a guarantee of a particular solution quality. Run the cells in order; the printed values and plots are produced by the code.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np

# Fixing both seeds makes the example reproducible while the optimizer also
# uses its own random-number generator for independent sensitivity runs.
np.random.seed(42)
random.seed(42)

plt.style.use("default")

In [ ]:
@dataclass
class Facility:
    """Data describing one terminal facility."""

    name: str
    area_requirement: float
    capacity: float
    priority: float


@dataclass
class Location:
    """Data describing one candidate site in the terminal plane."""

    id: int
    x: float
    y: float
    area: float


@dataclass
class Harmony:
    """A feasible assignment, its cost, and its maximization fitness."""

    assignment: List[int]
    cost: float
    fitness: float


# The example contains five facilities and eight candidate locations.
facilities = [
    Facility("Berth Area", 6000, 120, 1.0),
    Facility("Yard Block A", 4500, 90, 0.8),
    Facility("Yard Block B", 4500, 90, 0.8),
    Facility("Gate Complex", 2500, 70, 0.6),
    Facility("Maintenance", 1500, 40, 0.4),
]

locations = [
    Location(0, 0, 0, 8000),
    Location(1, 8, 0, 6000),
    Location(2, 16, 0, 7000),
    Location(3, 0, 8, 4000),
    Location(4, 8, 8, 5000),
    Location(5, 16, 8, 4500),
    Location(6, 4, 4, 5500),
    Location(7, 12, 4, 4800),
]

# Rows are origin facilities and columns are destination facilities.
flow_matrix = np.array(
    [
        [0, 60, 45, 25, 15],
        [55, 0, 30, 35, 20],
        [50, 25, 0, 30, 18],
        [20, 30, 25, 0, 10],
        [10, 15, 12, 8, 0],
    ],
    dtype=float,
)

transport_cost_rate = 0.025

assert flow_matrix.shape == (len(facilities), len(facilities))
assert np.all(flow_matrix >= 0)

print(f"Facilities: {[facility.name for facility in facilities]}")
print(f"Candidate locations: {len(locations)}")
print(f"Flow matrix shape: {flow_matrix.shape}")
print(f"Total directed flow: {flow_matrix.sum():.0f} containers/hour")

In [ ]:
def calculate_distance(loc1: Location, loc2: Location) -> float:
    """Return Euclidean distance between two sites, in meters."""
    return float(np.hypot(loc1.x - loc2.x, loc1.y - loc2.y))


def calculate_transport_cost(
    facility_i_idx: int,
    facility_j_idx: int,
    location_i_idx: int,
    location_j_idx: int,
    facilities: List[Facility],
    locations: List[Location],
    flow_matrix: np.ndarray,
    transport_cost_rate: float,
) -> float:
    """Return directed hourly cost for one facility-to-facility flow."""
    distance = calculate_distance(
        locations[location_i_idx], locations[location_j_idx]
    )
    flow = flow_matrix[facility_i_idx, facility_j_idx]
    return float(flow * distance * transport_cost_rate)


def calculate_total_cost(
    assignment: List[int],
    facilities: List[Facility],
    locations: List[Location],
    flow_matrix: np.ndarray,
    transport_cost_rate: float,
) -> float:
    """Sum directed transportation costs for a complete assignment."""
    total_cost = 0.0
    for i in range(len(facilities)):
        for j in range(len(facilities)):
            if i != j:
                total_cost += calculate_transport_cost(
                    i,
                    j,
                    assignment[i],
                    assignment[j],
                    facilities,
                    locations,
                    flow_matrix,
                    transport_cost_rate,
                )
    return total_cost


def is_feasible_assignment(
    assignment: List[int],
    facilities: List[Facility],
    locations: List[Location],
) -> bool:
    """Check assignment length, site uniqueness, and area requirements."""
    if len(assignment) != len(facilities):
        return False
    if any(index < 0 or index >= len(locations) for index in assignment):
        return False
    if len(set(assignment)) != len(assignment):
        return False
    return all(
        facility.area_requirement <= locations[location_idx].area
        for facility, location_idx in zip(facilities, assignment)
    )


def generate_random_feasible_assignment(
    facilities: List[Facility],
    locations: List[Location],
    rng: random.Random,
    max_attempts: int = 1000,
) -> Optional[List[int]]:
    """Try randomized sequential assignments and return a feasible one."""
    for _ in range(max_attempts):
        available_locations = list(range(len(locations)))
        rng.shuffle(available_locations)
        assignment = []

        for facility in facilities:
            feasible_locations = [
                location_idx
                for location_idx in available_locations
                if facility.area_requirement <= locations[location_idx].area
            ]
            if not feasible_locations:
                break

            chosen_location = rng.choice(feasible_locations)
            assignment.append(chosen_location)
            available_locations.remove(chosen_location)

        if is_feasible_assignment(assignment, facilities, locations):
            return assignment

    return None


sample_distance = calculate_distance(locations[0], locations[4])
sample_cost = calculate_transport_cost(
    0, 1, 0, 4, facilities, locations, flow_matrix, transport_cost_rate
)
print(f"Distance between Location 0 and Location 4: {sample_distance:.2f} m")
print(f"Berth-to-Yard cost from Location 0 to Location 4: ${sample_cost:.2f}/hour")

In [ ]:
class HarmonySearchOptimizer:
    """Harmony Search for the constrained terminal assignment problem."""

    def __init__(
        self,
        facilities: List[Facility],
        locations: List[Location],
        flow_matrix: np.ndarray,
        transport_cost_rate: float,
        hms: int = 20,
        hmcr: float = 0.9,
        par: float = 0.3,
        max_iterations: int = 1000,
        seed: int = 42,
    ) -> None:
        if hms < 2:
            raise ValueError("Harmony Memory Size must be at least 2.")
        if not 0.0 <= hmcr <= 1.0:
            raise ValueError("HMCR must be between 0 and 1.")
        if not 0.0 <= par <= 1.0:
            raise ValueError("PAR must be between 0 and 1.")
        if max_iterations < 1:
            raise ValueError("Maximum iterations must be at least 1.")
        if flow_matrix.shape != (len(facilities), len(facilities)):
            raise ValueError("The flow matrix must have one row and column per facility.")
        if len(locations) < len(facilities):
            raise ValueError("There must be at least as many locations as facilities.")
        if transport_cost_rate < 0:
            raise ValueError("The transportation cost rate cannot be negative.")

        self.facilities = facilities
        self.locations = locations
        self.flow_matrix = flow_matrix
        self.transport_cost_rate = transport_cost_rate
        self.hms = hms
        self.hmcr = hmcr
        self.par = par
        self.max_iterations = max_iterations
        self.seed = seed
        self.rng = random.Random(seed)

        self.harmony_memory: List[Harmony] = []
        self.best_harmony: Optional[Harmony] = None
        self.convergence_history: List[float] = []
        self.diversity_history: List[float] = []

    def initialize_harmony_memory(self) -> None:
        """Fill memory with independently generated feasible assignments."""
        self.harmony_memory = []
        self.convergence_history = []
        self.diversity_history = []
        print(f"Initializing Harmony Memory with {self.hms} solutions...")

        attempts = 0
        max_initialization_attempts = self.hms * 1000
        while len(self.harmony_memory) < self.hms:
            attempts += 1
            if attempts > max_initialization_attempts:
                raise RuntimeError(
                    "Unable to generate enough feasible assignments. "
                    "Check the site capacities and facility requirements."
                )

            assignment = generate_random_feasible_assignment(
                self.facilities, self.locations, self.rng
            )
            if assignment is None:
                continue

            cost = calculate_total_cost(
                assignment,
                self.facilities,
                self.locations,
                self.flow_matrix,
                self.transport_cost_rate,
            )
            if cost <= 0:
                raise ValueError("Fitness requires a strictly positive assignment cost.")
            self.harmony_memory.append(
                Harmony(assignment.copy(), cost, 1.0 / cost)
            )

        self.harmony_memory.sort(key=lambda harmony: harmony.fitness, reverse=True)
        self.best_harmony = self.harmony_memory[0]
        print(f"Initial best cost: ${self.best_harmony.cost:.2f}/hour")

    def improvise_new_harmony(self) -> List[int]:
        """Construct a feasible candidate using memory, adjustment, and chance."""
        for _ in range(100):
            new_assignment = []
            available_locations = set(range(len(self.locations)))

            for facility_idx, facility in enumerate(self.facilities):
                feasible_locations = [
                    location_idx
                    for location_idx in available_locations
                    if facility.area_requirement <= self.locations[location_idx].area
                ]
                if not feasible_locations:
                    break

                use_memory = self.rng.random() < self.hmcr
                if use_memory:
                    selected_harmony = self.rng.choice(self.harmony_memory)
                    candidate_location = selected_harmony.assignment[facility_idx]

                    adjust_pitch = self.rng.random() < self.par
                    if adjust_pitch:
                        candidate_location = self.rng.choice(feasible_locations)
                    elif candidate_location not in feasible_locations:
                        candidate_location = self.rng.choice(feasible_locations)
                else:
                    candidate_location = self.rng.choice(feasible_locations)

                new_assignment.append(candidate_location)
                available_locations.remove(candidate_location)

            if is_feasible_assignment(
                new_assignment, self.facilities, self.locations
            ):
                return new_assignment

        # A fresh feasible assignment preserves the constraints if the
        # sequential improvisation attempts cannot complete the candidate.
        fallback_assignment = generate_random_feasible_assignment(
            self.facilities, self.locations, self.rng
        )
        if fallback_assignment is None:
            raise RuntimeError("Unable to construct a feasible harmony.")
        return fallback_assignment

    def update_harmony_memory(self, new_harmony: Harmony) -> bool:
        """Replace the worst harmony only when the candidate is strictly better."""
        if new_harmony.fitness <= self.harmony_memory[-1].fitness:
            return False

        self.harmony_memory[-1] = new_harmony
        self.harmony_memory.sort(key=lambda harmony: harmony.fitness, reverse=True)
        if new_harmony.fitness > self.best_harmony.fitness:
            self.best_harmony = new_harmony
            return True
        return False

    def calculate_diversity(self) -> float:
        """Return mean pairwise Hamming distance across stored assignments."""
        if len(self.harmony_memory) < 2:
            return 0.0

        total_distance = 0
        comparisons = 0
        for i in range(len(self.harmony_memory)):
            for j in range(i + 1, len(self.harmony_memory)):
                total_distance += sum(
                    first != second
                    for first, second in zip(
                        self.harmony_memory[i].assignment,
                        self.harmony_memory[j].assignment,
                    )
                )
                comparisons += 1
        return total_distance / comparisons

    def optimize(self) -> Tuple[List[int], float, Dict]:
        """Run Harmony Search and return the best assignment and diagnostics."""
        start_time = time.perf_counter()
        self.initialize_harmony_memory()

        improvement_count = 0
        stagnation_count = 0
        print("\nStarting Harmony Search optimization...")
        print(f"Parameters: HMS={self.hms}, HMCR={self.hmcr}, PAR={self.par}")
        print(f"Maximum iterations: {self.max_iterations}")

        iteration = 0
        for iteration in range(self.max_iterations):
            new_assignment = self.improvise_new_harmony()
            if not is_feasible_assignment(
                new_assignment, self.facilities, self.locations
            ):
                raise AssertionError("Improvisation must produce a feasible assignment.")

            new_cost = calculate_total_cost(
                new_assignment,
                self.facilities,
                self.locations,
                self.flow_matrix,
                self.transport_cost_rate,
            )
            if new_cost <= 0:
                raise ValueError("Fitness requires a strictly positive assignment cost.")
            new_harmony = Harmony(new_assignment, new_cost, 1.0 / new_cost)

            improved = self.update_harmony_memory(new_harmony)
            if improved:
                improvement_count += 1
                stagnation_count = 0
            else:
                stagnation_count += 1

            self.convergence_history.append(self.best_harmony.cost)
            self.diversity_history.append(self.calculate_diversity())

            if (iteration + 1) % 100 == 0:
                print(
                    f"Iteration {iteration + 1}: "
                    f"best cost = ${self.best_harmony.cost:.2f}/hour, "
                    f"diversity = {self.diversity_history[-1]:.2f}"
                )

            if stagnation_count > 200:
                print(
                    f"Stopping at iteration {iteration + 1}: "
                    "no best-solution improvement for more than 200 iterations."
                )
                break

        execution_time = time.perf_counter() - start_time
        results = {
            "best_assignment": self.best_harmony.assignment.copy(),
            "best_cost": self.best_harmony.cost,
            "execution_time": execution_time,
            "iterations": iteration + 1,
            "improvements": improvement_count,
            "convergence_history": self.convergence_history.copy(),
            "diversity_history": self.diversity_history.copy(),
            "final_harmony_memory": self.harmony_memory.copy(),
        }
        return self.best_harmony.assignment.copy(), self.best_harmony.cost, results


print("Harmony Search optimizer is ready.")

In [ ]:
optimizer = HarmonySearchOptimizer(
    facilities=facilities,
    locations=locations,
    flow_matrix=flow_matrix,
    transport_cost_rate=transport_cost_rate,
    hms=20,
    hmcr=0.9,
    par=0.3,
    max_iterations=1000,
    seed=42,
)

best_assignment, best_cost, results = optimizer.optimize()

assert is_feasible_assignment(best_assignment, facilities, locations)
assert np.isclose(
    calculate_total_cost(
        best_assignment, facilities, locations, flow_matrix, transport_cost_rate
    ),
    best_cost,
)

print("\n" + "=" * 60)
print("HARMONY SEARCH RUN RESULTS")
print("=" * 60)
print(f"Elapsed time: {results['execution_time']:.3f} seconds")
print(f"Iterations completed: {results['iterations']}")
print(f"Best-solution improvements: {results['improvements']}")
print(f"Best modeled cost: ${best_cost:.2f}/hour")
print("\nBest feasible assignment found:")
for facility, location_idx in zip(facilities, best_assignment):
    location = locations[location_idx]
    print(
        f"  {facility.name}: Location {location_idx} "
        f"at ({location.x:g}, {location.y:g}) m"
    )

initial_best_cost = results["convergence_history"][0]
improvement_percent = (
    (initial_best_cost - best_cost) / initial_best_cost * 100
)
print(f"\nImprovement from initial best to final best: {improvement_percent:.1f}%")

In [ ]:
def analyze_harmony_memory(
    results: Dict,
    facilities: List[Facility],
) -> None:
    """Summarize costs and location usage in the final Harmony Memory."""
    harmony_memory = results["final_harmony_memory"]
    costs = [harmony.cost for harmony in harmony_memory]

    print("\n" + "=" * 50)
    print("HARMONY MEMORY DIAGNOSTICS")
    print("=" * 50)
    print(f"Stored feasible assignments: {len(harmony_memory)}")
    print(f"Best cost: ${min(costs):.2f}/hour")
    print(f"Worst cost: ${max(costs):.2f}/hour")
    print(f"Cost range: ${max(costs) - min(costs):.2f}/hour")
    print(f"Mean cost: ${np.mean(costs):.2f}/hour")
    print(f"Population standard deviation: ${np.std(costs):.2f}/hour")
    print(f"Median cost: ${np.median(costs):.2f}/hour")

    print("\nFive lowest-cost stored assignments:")
    for rank, harmony in enumerate(harmony_memory[:5], start=1):
        print(
            f"  {rank}. Cost: ${harmony.cost:.2f}/hour; "
            f"assignment: {harmony.assignment}"
        )

    location_counts = {location_idx: 0 for location_idx in range(len(locations))}
    for harmony in harmony_memory:
        for location_idx in harmony.assignment:
            location_counts[location_idx] += 1

    total_assignments = len(harmony_memory) * len(facilities)
    print("\nLocation usage across facility assignments in memory:")
    for location_idx, count in location_counts.items():
        frequency = 100 * count / total_assignments
        print(f"  Location {location_idx}: {count} uses ({frequency:.1f}%)")


analyze_harmony_memory(results, facilities)

In [ ]:
def summarize_harmony_search_run(results: Dict) -> Dict[str, float]:
    """Report measured diagnostics from this run without estimated benchmarks."""
    summary = {
        "best_cost_per_hour": results["best_cost"],
        "elapsed_seconds": results["execution_time"],
        "iterations": results["iterations"],
        "best_solution_improvements": results["improvements"],
        "final_memory_mean_cost_per_hour": float(
            np.mean([harmony.cost for harmony in results["final_harmony_memory"]])
        ),
        "final_memory_diversity": results["diversity_history"][-1],
    }

    print("\n" + "=" * 55)
    print("MEASURED RUN DIAGNOSTICS")
    print("=" * 55)
    print(f"Best modeled cost: ${summary['best_cost_per_hour']:.2f}/hour")
    print(f"Elapsed time: {summary['elapsed_seconds']:.3f} seconds")
    print(f"Iterations completed: {summary['iterations']}")
    print(f"Best-solution improvements: {summary['best_solution_improvements']}")
    print(
        "Final memory mean cost: "
        f"${summary['final_memory_mean_cost_per_hour']:.2f}/hour"
    )
    print(f"Final mean pairwise Hamming distance: {summary['final_memory_diversity']:.2f}")
    print(
        "These measurements describe this seeded run and machine; "
        "they are not a comparison with an exact optimum or another algorithm."
    )
    return summary


run_summary = summarize_harmony_search_run(results)

In [ ]:
def visualize_harmony_search_results(
    facilities: List[Facility],
    locations: List[Location],
    best_assignment: List[int],
    results: Dict,
) -> None:
    """Plot the assignment and the search's convergence and diversity."""
    fig, (ax_layout, ax_convergence, ax_diversity) = plt.subplots(
        1, 3, figsize=(18, 5)
    )

    assigned_locations = set(best_assignment)
    ax_layout.scatter(
        [location.x for location in locations],
        [location.y for location in locations],
        marker="x",
        color="gray",
        s=70,
        label="Candidate site",
    )
    colors = plt.get_cmap("tab10")
    for facility_idx, (facility, location_idx) in enumerate(
        zip(facilities, best_assignment)
    ):
        location = locations[location_idx]
        ax_layout.scatter(
            location.x,
            location.y,
            s=110,
            color=colors(facility_idx),
            edgecolor="black",
            label=facility.name,
            zorder=3,
        )
        ax_layout.annotate(
            f"{facility.name}\nSite {location_idx}",
            (location.x, location.y),
            xytext=(5, 7),
            textcoords="offset points",
            fontsize=8,
        )

    for location in locations:
        if location.id not in assigned_locations:
            ax_layout.annotate(
                f"Site {location.id}",
                (location.x, location.y),
                xytext=(5, -12),
                textcoords="offset points",
                fontsize=8,
                color="gray",
            )
    ax_layout.set_title("Best Feasible Assignment Found")
    ax_layout.set_xlabel("X coordinate (m)")
    ax_layout.set_ylabel("Y coordinate (m)")
    ax_layout.set_aspect("equal", adjustable="datalim")
    ax_layout.grid(True, alpha=0.3)
    ax_layout.legend(fontsize=7, loc="best")

    iterations = np.arange(1, len(results["convergence_history"]) + 1)
    ax_convergence.plot(
        iterations,
        results["convergence_history"],
        color="tab:blue",
        linewidth=1.5,
    )
    ax_convergence.set_title("Best Cost Recorded")
    ax_convergence.set_xlabel("Iteration")
    ax_convergence.set_ylabel("Modeled cost ($/hour)")
    ax_convergence.grid(True, alpha=0.3)

    ax_diversity.plot(
        iterations,
        results["diversity_history"],
        color="tab:green",
        linewidth=1.5,
    )
    ax_diversity.set_title("Harmony Memory Diversity")
    ax_diversity.set_xlabel("Iteration")
    ax_diversity.set_ylabel("Mean pairwise Hamming distance")
    ax_diversity.grid(True, alpha=0.3)

    fig.suptitle(
        f"Harmony Search diagnostics (best cost: ${results['best_cost']:.2f}/hour)",
        fontweight="bold",
    )
    fig.tight_layout()
    plt.show()


visualize_harmony_search_results(facilities, locations, best_assignment, results)

In [ ]:
def parameter_sensitivity_analysis(
    facilities: List[Facility],
    locations: List[Location],
    flow_matrix: np.ndarray,
    transport_cost_rate: float,
) -> Dict[str, List[float]]:
    """Explore how one seeded run responds to selected HS parameters."""
    print("\n" + "=" * 50)
    print("PARAMETER SENSITIVITY ANALYSIS")
    print("=" * 50)
    print(
        "Each setting uses one run with seed 42 and 500 maximum iterations. "
        "The results illustrate this instance; they are not statistical guarantees."
    )

    hms_values = [10, 20, 30, 40]
    hmcr_values = [0.7, 0.8, 0.9, 0.95]
    par_values = [0.1, 0.2, 0.3, 0.4, 0.5]
    hms_results = []
    hmcr_results = []
    par_results = []

    parameter_groups = [
        ("Harmony Memory Size (HMS)", hms_values, hms_results, "hms"),
        ("Harmony Memory Consideration Rate (HMCR)", hmcr_values, hmcr_results, "hmcr"),
        ("Pitch Adjustment Rate (PAR)", par_values, par_results, "par"),
    ]

    for label, values, costs, varied_parameter in parameter_groups:
        print(f"\nTesting {label}:")
        for value in values:
            parameters = {"hms": 20, "hmcr": 0.9, "par": 0.3}
            parameters[varied_parameter] = value
            test_optimizer = HarmonySearchOptimizer(
                facilities=facilities,
                locations=locations,
                flow_matrix=flow_matrix,
                transport_cost_rate=transport_cost_rate,
                hms=parameters["hms"],
                hmcr=parameters["hmcr"],
                par=parameters["par"],
                max_iterations=500,
                seed=42,
            )
            _, test_cost, test_results = test_optimizer.optimize()
            costs.append(test_cost)
            print(
                f"  {varied_parameter.upper()}={value}: "
                f"best cost=${test_cost:.2f}/hour, "
                f"elapsed={test_results['execution_time']:.3f} seconds"
            )

    fig, (ax_hms, ax_hmcr, ax_par) = plt.subplots(1, 3, figsize=(15, 5))
    for axis, values, costs, title, color, xlabel in [
        (ax_hms, hms_values, hms_results, "HMS sensitivity", "tab:blue", "HMS"),
        (ax_hmcr, hmcr_values, hmcr_results, "HMCR sensitivity", "tab:red", "HMCR"),
        (ax_par, par_values, par_results, "PAR sensitivity", "tab:green", "PAR"),
    ]:
        axis.plot(values, costs, "o-", color=color)
        axis.set_xlabel(xlabel)
        axis.set_ylabel("Best modeled cost ($/hour)")
        axis.set_title(title)
        axis.grid(True, alpha=0.3)

    fig.suptitle("One-run Harmony Search parameter sensitivity", fontweight="bold")
    fig.tight_layout()
    plt.show()

    best_settings = {
        "hms": hms_values[int(np.argmin(hms_results))],
        "hmcr": hmcr_values[int(np.argmin(hmcr_results))],
        "par": par_values[int(np.argmin(par_results))],
    }
    print("\nLowest-cost tested setting for each one-parameter sweep:")
    for parameter, value in best_settings.items():
        print(f"  {parameter.upper()}: {value}")
    print(
        "These separate sweeps do not establish a jointly optimal parameter set. "
        "Repeated runs would be needed to assess variability across random seeds."
    )

    return {
        "hms_values": hms_values,
        "hms_costs": hms_results,
        "hmcr_values": hmcr_values,
        "hmcr_costs": hmcr_results,
        "par_values": par_values,
        "par_costs": par_results,
    }


sensitivity_results = parameter_sensitivity_analysis(
    facilities, locations, flow_matrix, transport_cost_rate
)

## Interpreting the diagnostics

The assignment plot shows which candidate site each facility uses; point size is only a plotting symbol and does not represent a physical footprint. The convergence plot is the best cost recorded so far, so it cannot increase during a run. A flat segment means that no better best-so-far assignment was recorded during those iterations; it does not prove that the global optimum has been found.

The diversity plot is the mean Hamming distance among stored assignments. A decrease can indicate that the population is becoming more similar, while a larger value indicates more variation in facility-to-site choices. Neither value alone establishes solution quality. The memory cost summary describes the assignments retained at the end, not a sample of all feasible layouts.

The sensitivity experiment changes one parameter at a time while holding the others at their stated baseline values. Each point is one seeded run, so observed differences are specific to this instance and seed. A stochastic method can return different results under different seeds; repeated runs are needed to characterize that variability. No exact optimum or cross-method performance comparison is computed in this notebook.

## Summary

This notebook applies Harmony Search to a constrained facility-to-site assignment problem. The implementation:

1. Represents each candidate layout as a complete assignment of facilities to distinct sites.
2. Rejects assignments that violate site uniqueness or area requirements.
3. Evaluates each feasible layout using directed flow, Euclidean distance, and the stated transportation cost rate.
4. Maintains a ranked Harmony Memory and generates candidates through memory consideration, pitch adjustment, and random selection.
5. Tracks the best-so-far cost and the mean pairwise Hamming distance in memory.
6. Examines the final memory and performs one-run, one-parameter-at-a-time sensitivity experiments.

Harmony Search is a heuristic search procedure, not a proof of optimality. Its output should be interpreted as the best feasible assignment encountered under the chosen seed, parameter settings, and stopping rule. The model's conclusions are also limited by its assumptions: only area feasibility and distinct site use are enforced, and the cost captures directed flow-distance expense rather than every cost or operational consideration in a real terminal.